<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px; float: right;">
    </div>
</a>

# **Factory Optimization with cuOpt**

<h2><b>Use Case 1:</b> Autoclave Formulation Extensions</h2>

This notebook is a formulation lab.

Notebook `01` builds the clean base model. This notebook shows code patterns for the next requirements we may add later.

The goal is not to turn every idea into the main workshop yet. The goal is to see exactly what would change in the data, variables, constraints, and objective.

<hr>

## What We Have Locked

For the autoclave use case, we are treating the problem as a **MIP scheduling / batching problem**.

The base model is:

> Choose which parts go into which autoclave runs.

The drill problem is separate and will be handled with routing/TSP/VRP later.

## How To Use This Notebook With `01`

Each extension below has two pieces:

1. A short explanation of the formulation change.
2. A runnable code cell that shows the data or model pattern.

When we decide an extension belongs in the main lesson, copy that code pattern into the matching section of `01-Autoclave-MIP-Workshop.ipynb`.

## Setup

Load the same autoclave data used in notebook `01`.

We also derive `recipe_id` the same way, so the examples stay aligned.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from cuopt.linear_programming.problem import CONTINUOUS, INTEGER, MINIMIZE, Problem

DATA_DIR = Path("data")
END_HR = 30

parts = pd.read_csv(DATA_DIR / "autoclave_parts.csv")
autoclaves = pd.read_csv(DATA_DIR / "autoclaves.csv")

parts["recipe_id"] = (
    parts["material"]
    + "_cure_"
    + parts["cure_time_hr"].map(lambda value: f"{value:g}h")
)

recipes = sorted(parts["recipe_id"].unique())
recipe_duration_hr = parts.groupby("recipe_id")["cure_time_hr"].first().to_dict()

print(f"Loaded {len(parts)} parts")
print(f"Loaded {len(autoclaves)} autoclaves")
print(f"Recipes: {recipes}")

display(parts.head())
display(autoclaves)

## Helper Functions

These helpers let us preview formulation changes without rebuilding the full notebook every time.

In [ ]:
def safe_name(value):
    return str(value).replace("-", "_").replace(".", "p").replace(" ", "_")


def build_candidate_runs(active_autoclaves, recipe_ids, end_hr=24):
    rows = []

    for _, autoclave in active_autoclaves.iterrows():
        run_starts = range(
            int(autoclave.available_from_hr),
            end_hr + 1,
            int(autoclave.slot_duration_hr),
        )

        for recipe_id in recipe_ids:
            for run_start in run_starts:
                rows.append({
                    "autoclave_id": autoclave.autoclave_id,
                    "recipe_id": recipe_id,
                    "start_hr": int(run_start),
                    "finish_hr": int(run_start) + float(recipe_duration_hr[recipe_id]),
                    "batch_capacity": int(autoclave.batch_capacity),
                    "usable_area_m2": float(autoclave.usable_area_m2),
                })

    return pd.DataFrame(rows)


def build_assignment_choices(
    parts_df,
    candidate_runs_df,
    allow_late=False,
    earliest_start_col="ready_hr",
):
    rows = []

    for _, part in parts_df.iterrows():
        for _, run in candidate_runs_df.iterrows():
            if part.recipe_id != run.recipe_id:
                continue
            if float(run.start_hr) < float(part[earliest_start_col]):
                continue

            finish_hr = float(run.start_hr) + float(part.cure_time_hr)
            lateness_hr = max(0, finish_hr - float(part.deadline_hr))

            if not allow_late and lateness_hr > 0:
                continue

            rows.append({
                "part_id": part.part_id,
                "autoclave_id": run.autoclave_id,
                "recipe_id": part.recipe_id,
                "start_hr": int(run.start_hr),
                "finish_hr": finish_hr,
                "deadline_hr": float(part.deadline_hr),
                "lateness_hr": lateness_hr,
                "priority": int(part.priority),
                "area_m2": float(part.area_m2),
            })

    return pd.DataFrame(rows)


active_ac1 = autoclaves.query("autoclave_id == 'AC-1'").copy()
candidate_runs_ac1 = build_candidate_runs(active_ac1, recipes, end_hr=24)
hard_deadline_choices = build_assignment_choices(
    parts,
    candidate_runs_ac1,
    allow_late=False,
)
soft_deadline_choices = build_assignment_choices(
    parts,
    candidate_runs_ac1,
    allow_late=True,
)

print(f"Candidate runs for AC-1: {len(candidate_runs_ac1)}")
print(f"Hard-deadline assignment choices: {len(hard_deadline_choices)}")
print(f"Soft-deadline assignment choices: {len(soft_deadline_choices)}")

## Base Model Recap: Hard-Deadline Model

The baseline hard-deadline model answers:

> What parts can we cure before the out-time deadline if capacity is limited?

This is a **hard-deadline** model.

Notebook `01` uses this rule when it creates assignment variables:

```python
if finish_hr > deadline_hr:
    continue
```

That means the solver never sees an assignment that would finish after out-time.

In [ ]:
choice_summary = (
    hard_deadline_choices.groupby("part_id")
    .agg(
        hard_deadline_choices=("start_hr", "count"),
        earliest_hard_start_hr=("start_hr", "min"),
    )
    .reset_index()
    .merge(
        soft_deadline_choices.groupby("part_id")
        .agg(
            soft_deadline_choices=("start_hr", "count"),
            best_possible_lateness_hr=("lateness_hr", "min"),
        )
        .reset_index(),
        on="part_id",
        how="outer",
    )
    .merge(parts[["part_id", "part_type", "priority", "deadline_hr"]], on="part_id")
    .sort_values(["deadline_hr", "priority"], ascending=[True, False])
)

display(choice_summary)

<hr>

## Extension 1: Add More Autoclaves

This is now implemented at the end of notebook `01`.

The decision variable changes from:

> `x[part, recipe, start]`

into:

> `x[part, autoclave, recipe, start]`

### Where This Changes In `01`

Change these sections:

1. `Prepare The First Scenario`
2. `Step 2: Define Candidate Autoclave Runs`
3. `Step 3: Add Variables`
4. `Step 4: Add Constraints`
5. `Step 7: Analyze The Results`

In [ ]:
scenario_rows = []

for scenario_name, active_autoclaves in [
    ("AC-1 only", autoclaves.query("autoclave_id == 'AC-1'").copy()),
    ("AC-1 + AC-2", autoclaves.copy()),
]:
    candidate_runs = build_candidate_runs(active_autoclaves, recipes, end_hr=24)
    assignment_choices = build_assignment_choices(parts, candidate_runs, allow_late=False)

    scenario_rows.append({
        "scenario": scenario_name,
        "autoclaves": active_autoclaves["autoclave_id"].nunique(),
        "candidate_runs": len(candidate_runs),
        "assignment_variables_available": len(assignment_choices),
        "parts_with_at_least_one_choice": assignment_choices["part_id"].nunique(),
    })

scenario_comparison = pd.DataFrame(scenario_rows)

display(scenario_comparison)

two_ac_candidate_runs = build_candidate_runs(autoclaves, recipes, end_hr=24)
display(two_ac_candidate_runs.head(12))

## Extension 2: Soft-Deadline Model, Schedule Every Part

The soft-deadline extension answers a different question:

> If every part must be scheduled, what is the least painful schedule?

This changes the model in two important ways:

1. Each part must be assigned exactly once.
2. Late finishes are allowed, but the objective penalizes lateness.

Because `deadline_hr` is an out-time deadline in this workshop, this is not saying late cure is automatically acceptable. It is a recovery/escalation model.

### Where This Changes In `01`

Change these sections:

1. `Step 3: Add Variables` - allow assignments after `deadline_hr`.
2. `Constraint 1` - change `<= 1` to `== 1`.
3. `Step 4` - add lateness variables and lateness constraints.
4. `Step 5` - minimize weighted lateness.
5. `Step 7` - show late parts and lateness hours.

In [ ]:
def solve_soft_deadline_extension(parts_df, active_autoclave, end_hr=30):
    problem = Problem("Autoclave Soft Deadline Demo")

    run_starts = list(range(
        int(active_autoclave.available_from_hr),
        end_hr + 1,
        int(active_autoclave.slot_duration_hr),
    ))
    recipe_ids = sorted(parts_df["recipe_id"].unique())

    run_used = {}
    for recipe_id in recipe_ids:
        for run_start in run_starts:
            run_used[(recipe_id, run_start)] = problem.addVariable(
                lb=0,
                ub=1,
                vtype=INTEGER,
                name=f"y_{safe_name(recipe_id)}_{run_start}",
            )

    assign = {}
    choice_finish = {}
    for _, part in parts_df.iterrows():
        for run_start in run_starts:
            finish_hr = run_start + float(part.cure_time_hr)

            if run_start < float(part.ready_hr):
                continue

            key = (part.part_id, part.recipe_id, int(run_start))
            assign[key] = problem.addVariable(
                lb=0,
                ub=1,
                vtype=INTEGER,
                name=f"x_{safe_name(part.part_id)}_{safe_name(part.recipe_id)}_{run_start}",
            )
            choice_finish[key] = finish_hr

    lateness = {}
    for _, part in parts_df.iterrows():
        part_id = part.part_id
        choices = [
            variable
            for (candidate_part_id, recipe_id, run_start), variable in assign.items()
            if candidate_part_id == part_id
        ]

        problem.addConstraint(
            sum(choices) == 1,
            name=f"exactly_once_{safe_name(part_id)}",
        )

        lateness[part_id] = problem.addVariable(
            lb=0,
            ub=100,
            vtype=CONTINUOUS,
            name=f"lateness_{safe_name(part_id)}",
        )

        finish_expr = sum(
            choice_finish[(candidate_part_id, recipe_id, run_start)] * variable
            for (candidate_part_id, recipe_id, run_start), variable in assign.items()
            if candidate_part_id == part_id
        )

        problem.addConstraint(
            lateness[part_id] >= finish_expr - float(part.deadline_hr),
            name=f"lateness_lb_{safe_name(part_id)}",
        )

    run_capacity = int(active_autoclave.batch_capacity)
    for recipe_id in recipe_ids:
        for run_start in run_starts:
            choices = [
                variable
                for (part_id, candidate_recipe_id, candidate_start), variable in assign.items()
                if candidate_recipe_id == recipe_id and candidate_start == run_start
            ]
            y = run_used[(recipe_id, run_start)]

            problem.addConstraint(
                sum(choices) <= run_capacity * y,
                name=f"capacity_{safe_name(recipe_id)}_{run_start}",
            )

            if choices:
                problem.addConstraint(
                    y <= sum(choices),
                    name=f"activate_only_if_used_{safe_name(recipe_id)}_{run_start}",
                )
            else:
                problem.addConstraint(
                    y == 0,
                    name=f"no_feasible_parts_{safe_name(recipe_id)}_{run_start}",
                )

    for run_start in run_starts:
        problem.addConstraint(
            sum(run_used[(recipe_id, run_start)] for recipe_id in recipe_ids) <= 1,
            name=f"one_recipe_at_start_{run_start}",
        )

    priority_by_part = parts_df.set_index("part_id")["priority"].to_dict()
    weighted_lateness = sum(
        (10 + priority_by_part[part_id]) * late_var
        for part_id, late_var in lateness.items()
    )
    early_finish_tiebreaker = sum(
        0.001 * choice_finish[key] * variable
        for key, variable in assign.items()
    )

    problem.setObjective(weighted_lateness + early_finish_tiebreaker, sense=MINIMIZE)
    problem.solve()

    solution_rows = []
    for (part_id, recipe_id, run_start), variable in assign.items():
        if variable.Value > 0.5:
            part = parts_df.query("part_id == @part_id").iloc[0]
            finish_hr = choice_finish[(part_id, recipe_id, run_start)]
            solution_rows.append({
                "part_id": part_id,
                "part_type": part.part_type,
                "recipe_id": recipe_id,
                "start_hr": run_start,
                "finish_hr": finish_hr,
                "deadline_hr": float(part.deadline_hr),
                "lateness_hr": max(0, finish_hr - float(part.deadline_hr)),
                "priority": int(part.priority),
            })

    solution = pd.DataFrame(solution_rows).sort_values(
        ["start_hr", "recipe_id", "priority"],
        ascending=[True, True, False],
    )

    return problem, solution

In [ ]:
active_autoclave = autoclaves.query("autoclave_id == 'AC-1'").iloc[0]
soft_deadline_problem, soft_deadline_solution = solve_soft_deadline_extension(
    parts,
    active_autoclave,
    end_hr=30,
)

late_parts = soft_deadline_solution.query("lateness_hr > 0").copy()

print("|============================================================|")
print("|                  Soft-Deadline Solution Metadata                |")
print("|============================================================|")
print(f"Objective value: {soft_deadline_problem.ObjValue:.3f}")
print(f"Solve time: {soft_deadline_problem.SolveTime:.3f} seconds")
print(f"Scheduled parts: {len(soft_deadline_solution)} of {len(parts)}")
print(f"Late parts: {len(late_parts)}")
print(f"Total lateness: {soft_deadline_solution['lateness_hr'].sum():.1f} hours")

print("\nSoft-deadline schedule:")
display(soft_deadline_solution)

print("Late parts:")
display(late_parts[[
    "part_id", "part_type", "start_hr", "finish_hr",
    "deadline_hr", "lateness_hr", "priority",
]])

## Extension 3: Hard Deadline Or Soft Target

The difference between the hard-deadline and soft-deadline models is not the data. It is how the model treats `deadline_hr`.

| Requirement | Variable creation | Assignment constraint | Objective |
|---|---|---|---|
| Hard out-time deadline | remove late choices | `<= 1` | maximize cured parts |
| Soft recovery target | keep late choices | `== 1` | minimize lateness |

Use this table when deciding whether a customer requirement is a hard factory rule or a business tradeoff.

In [ ]:
deadline_policy = pd.DataFrame([
    {
        "policy": "Hard out-time deadline",
        "variable_filter": "Skip choices where finish_hr > deadline_hr",
        "part_assignment_rule": "sum(part choices) <= 1",
        "objective": "maximize cured parts before deadline",
    },
    {
        "policy": "Soft recovery target",
        "variable_filter": "Keep late choices",
        "part_assignment_rule": "sum(part choices) == 1",
        "objective": "minimize weighted lateness",
    },
])

display(deadline_policy)

## Extension 4: Recipe Compatibility

The base model assumes exact recipe matching:

> same recipe can cure together; different recipe cannot.

A richer model can replace exact matching with a compatibility table.

### Where This Changes In `01`

Change these sections:

1. `Prepare The First Scenario` - load or create compatibility data.
2. `Step 2` - build candidate recipe groups.
3. `Step 3` - only create assignment variables for compatible parts.
4. `Constraint 3` - enforce one compatible recipe group per run.

In [ ]:
recipe_compatibility = pd.DataFrame(
    [
        {
            "recipe_a": recipe_a,
            "recipe_b": recipe_b,
            "can_cure_together": int(recipe_a == recipe_b),
        }
        for recipe_a in recipes
        for recipe_b in recipes
    ]
)

compatibility_matrix = recipe_compatibility.pivot(
    index="recipe_a",
    columns="recipe_b",
    values="can_cure_together",
)

display(compatibility_matrix)

compatible_recipe_groups = pd.DataFrame([
    {"run_group_id": recipe_id, "allowed_recipe_id": recipe_id}
    for recipe_id in recipes
])

print("Candidate recipe groups under the current exact-match rule:")
display(compatible_recipe_groups)

## Extension 5: Material Availability

Material availability may not change the model shape. It may only change the earliest feasible start time.

The base model checks:

```python
run_start >= ready_hr
```

The material-aware model checks:

```python
run_start >= max(ready_hr, material_ready_hr)
```

### Where This Changes In `01`

Change these sections:

1. `Prepare The First Scenario` - add `earliest_start_hr`.
2. `Step 3` - use `earliest_start_hr` instead of `ready_hr`.
3. `Step 7` - optionally show which parts were delayed by material.

In [ ]:
material_ready_demo = pd.DataFrame([
    {
        "material": "HexPly_8552",
        "material_ready_hr": 4,
        "note": "Demo assumption only; replace with real material data.",
    }
])

parts_with_material = parts.merge(material_ready_demo, on="material", how="left")
parts_with_material["material_ready_hr"] = parts_with_material["material_ready_hr"].fillna(0)
parts_with_material["earliest_start_hr"] = parts_with_material[[
    "ready_hr",
    "material_ready_hr",
]].max(axis=1)

material_aware_choices = build_assignment_choices(
    parts_with_material,
    candidate_runs_ac1,
    allow_late=False,
    earliest_start_col="earliest_start_hr",
)

material_comparison = (
    hard_deadline_choices.groupby("part_id").size().rename("base_choices")
    .to_frame()
    .join(material_aware_choices.groupby("part_id").size().rename("material_aware_choices"))
    .fillna(0)
    .astype(int)
    .reset_index()
    .merge(parts_with_material[[
        "part_id", "ready_hr", "material_ready_hr", "earliest_start_hr",
    ]], on="part_id")
)

print("Demo material rule: HexPly_8552 is not ready until hour 4")
display(material_comparison.head(10))

## Extension 6: Capacity By Footprint Instead Of Count

The base model uses:

> number of assigned parts <= batch capacity

The footprint model uses:

> total assigned area <= usable autoclave area

### Where This Changes In `01`

Change these sections:

1. `Prepare The First Scenario` - load `area_m2` and `usable_area_m2`.
2. `Constraint 2` - replace count capacity with area capacity.
3. `Step 7` - show area used by each run.

In [ ]:
area_preview = (
    hard_deadline_choices.groupby(["recipe_id", "start_hr"])
    .agg(
        candidate_parts=("part_id", "count"),
        candidate_area_m2=("area_m2", "sum"),
    )
    .reset_index()
    .assign(
        batch_capacity=int(active_autoclave.batch_capacity),
        usable_area_m2=float(active_autoclave.usable_area_m2),
    )
)
area_preview["over_count_capacity_if_all_chosen"] = (
    area_preview["candidate_parts"] > area_preview["batch_capacity"]
)
area_preview["over_area_capacity_if_all_chosen"] = (
    area_preview["candidate_area_m2"] > area_preview["usable_area_m2"]
)

display(area_preview.head(12))

print("Drop-in constraint pattern:")
print("sum(part_area[part_id] * x[part_id, recipe_id, start_hr]) <= usable_area_m2 * y[recipe_id, start_hr]")

## Extension 7: Setup Or Changeover Time

Setup time matters when changing from one recipe to another.

This is more advanced because the model needs to reason about neighboring runs.

### Where This Changes In `01`

Change these sections:

1. `Step 2` - make run order explicit.
2. `Step 4` - add transition or spacing constraints.
3. `Step 7` - show setup/idle time between runs.

In [ ]:
setup_time = pd.DataFrame(
    [
        {
            "previous_recipe": previous_recipe,
            "next_recipe": next_recipe,
            "setup_hr": 0 if previous_recipe == next_recipe else 1,
        }
        for previous_recipe in recipes
        for next_recipe in recipes
    ]
)

setup_matrix = setup_time.pivot(
    index="previous_recipe",
    columns="next_recipe",
    values="setup_hr",
)

display(setup_matrix)

run_sequence_demo = pd.DataFrame({
    "run_number": [1, 2, 3],
    "start_hr": [0, 3, 6],
    "recipe_id": [recipes[1], recipes[1], recipes[2]],
})
run_sequence_demo["previous_recipe"] = run_sequence_demo["recipe_id"].shift(1)
run_sequence_demo = run_sequence_demo.merge(
    setup_time,
    left_on=["previous_recipe", "recipe_id"],
    right_on=["previous_recipe", "next_recipe"],
    how="left",
)
run_sequence_demo["setup_hr"] = run_sequence_demo["setup_hr"].fillna(0)

display(run_sequence_demo[[
    "run_number", "start_hr", "previous_recipe", "recipe_id", "setup_hr",
]])

## Extension 8: Maintenance Downtime

Maintenance downtime is usually one of the cleanest extensions.

You can either remove blocked candidate runs before building variables, or keep them and force their run variable to zero.

### Where This Changes In `01`

Change these sections:

1. `Step 2` - remove candidate runs that overlap maintenance.
2. Or `Step 4` - add `y == 0` for blocked runs.
3. `Step 7` - show which time windows were unavailable.

In [ ]:
maintenance_windows = pd.DataFrame([
    {
        "autoclave_id": "AC-1",
        "downtime_start_hr": 9,
        "downtime_end_hr": 12,
        "reason": "Demo planned maintenance",
    }
])


def overlaps_window(run_start, run_finish, window_start, window_finish):
    return run_start < window_finish and run_finish > window_start


runs_with_maintenance = candidate_runs_ac1.copy()
runs_with_maintenance["blocked_by_maintenance"] = False

for _, window in maintenance_windows.iterrows():
    same_autoclave = runs_with_maintenance["autoclave_id"] == window.autoclave_id
    overlaps = runs_with_maintenance.apply(
        lambda row: overlaps_window(
            float(row.start_hr),
            float(row.finish_hr),
            float(window.downtime_start_hr),
            float(window.downtime_end_hr),
        ),
        axis=1,
    )
    runs_with_maintenance.loc[same_autoclave & overlaps, "blocked_by_maintenance"] = True

print("Blocked candidate runs:")
display(runs_with_maintenance.query("blocked_by_maintenance").head(12))

print("Candidate run count before and after filtering:")
display(pd.DataFrame([
    {"case": "before maintenance filter", "candidate_runs": len(candidate_runs_ac1)},
    {"case": "after maintenance filter", "candidate_runs": len(runs_with_maintenance.query("not blocked_by_maintenance"))},
]))

## Extension 9: Staging Space

Staging space means only so many parts can wait near the autoclave before loading.

We should not put this in the base model unless the customer gives us data. But the code shape is a time-bucket capacity rule.

### Where This Changes In `01`

Change these sections:

1. `Prepare The First Scenario` - add staging assumptions.
2. `Step 4` - add a capacity rule for each staging time bucket.
3. `Step 7` - plot staging usage.

In [ ]:
staging_lead_hr = 1
staging_capacity_parts = 4

staging_intervals = soft_deadline_solution[["part_id", "start_hr"]].copy()
staging_intervals["enter_staging_hr"] = (staging_intervals["start_hr"] - staging_lead_hr).clip(lower=0)
staging_intervals["leave_staging_hr"] = staging_intervals["start_hr"]

usage_rows = []
for hour in range(0, int(soft_deadline_solution["start_hr"].max()) + 1):
    staged_parts = staging_intervals[
        (staging_intervals["enter_staging_hr"] <= hour)
        & (staging_intervals["leave_staging_hr"] > hour)
    ]
    usage_rows.append({
        "hour": hour,
        "parts_in_staging": len(staged_parts),
        "staging_capacity_parts": staging_capacity_parts,
        "over_capacity": len(staged_parts) > staging_capacity_parts,
    })

staging_usage = pd.DataFrame(usage_rows)

display(staging_intervals.head(10))
display(staging_usage)

## Extension 10: Labor Or Crew Availability

People may be needed to load and unload an autoclave run.

This can be modeled like another time-bucket capacity rule.

### Where This Changes In `01`

Change these sections:

1. `Prepare The First Scenario` - load crew availability and crew requirements.
2. `Step 4` - add labor capacity constraints.
3. `Step 7` - show load/unload crew usage.

In [ ]:
crew_required_per_run = 2

runs_from_soft_deadline = (
    soft_deadline_solution.groupby(["start_hr", "recipe_id"])
    .size()
    .rename("parts_loaded")
    .reset_index()
)
runs_from_soft_deadline["crew_required"] = crew_required_per_run

crew_availability = pd.DataFrame({
    "hour": range(0, END_HR + 1),
    "crew_available": [2 if 6 <= hour <= 18 else 1 for hour in range(0, END_HR + 1)],
})

crew_usage = (
    runs_from_soft_deadline.groupby("start_hr")["crew_required"]
    .sum()
    .rename("crew_required")
    .reset_index()
    .rename(columns={"start_hr": "hour"})
    .merge(crew_availability, on="hour", how="right")
    .fillna({"crew_required": 0})
)
crew_usage["over_capacity"] = crew_usage["crew_required"] > crew_usage["crew_available"]

print("Runs that need loading crew:")
display(runs_from_soft_deadline)

print("Crew usage by hour:")
display(crew_usage.query("crew_required > 0 or over_capacity"))

## Extension 11: Objective Swaps

The same constraints can produce different schedules when we change the objective.

For teaching, this is useful because it shows that optimization is not just feasibility. It is also how we define **best**.

### Where This Changes In `01`

Change these sections:

1. `Step 5` - replace the objective expression and sense.
2. `Step 7` - add the metrics needed to compare schedules.

In [ ]:
objective_options = pd.DataFrame([
    {
        "objective_name": "Maximize cured parts",
        "sense": "MAXIMIZE",
        "code_pattern": "sum((1000 + priority[p]) * x[p, r, t])",
        "when_to_use": "Baseline hard-deadline model with limited capacity",
    },
    {
        "objective_name": "Minimize weighted lateness",
        "sense": "MINIMIZE",
        "code_pattern": "sum((10 + priority[p]) * lateness[p])",
        "when_to_use": "Soft-deadline model: schedule every part and quantify lateness",
    },
    {
        "objective_name": "Minimize extra equipment use",
        "sense": "MINIMIZE",
        "code_pattern": "sum(extra_autoclave_used[a])",
        "when_to_use": "When management wants to know whether extra capacity is worth it",
    },
    {
        "objective_name": "Minimize total area slack",
        "sense": "MINIMIZE",
        "code_pattern": "sum(unused_area[run])",
        "when_to_use": "When packing efficiency matters more than count capacity",
    },
])

display(objective_options)

soft_deadline_metrics = pd.DataFrame([
    {
        "metric": "scheduled_parts",
        "value": len(soft_deadline_solution),
    },
    {
        "metric": "late_parts",
        "value": len(soft_deadline_solution.query("lateness_hr > 0")),
    },
    {
        "metric": "total_lateness_hr",
        "value": soft_deadline_solution["lateness_hr"].sum(),
    },
])

display(soft_deadline_metrics)

<hr>

## Extension Tracker

This table keeps the candidate extensions in one place.

In [ ]:
extensions = pd.DataFrame([
    {
        "extension": "Add more autoclaves",
        "where_to_change_in_01": "Prepare scenario; Step 2 candidate runs; Step 3 variables; Step 4 constraints; Step 7 output",
        "model_change": "Add autoclave dimension to assignment variables",
        "data_needed": "autoclave_id, capacity, availability",
        "status": "Implemented in notebook 01; previewed here",
    },
    {
        "extension": "Soft-deadline model: schedule every part",
        "where_to_change_in_01": "Step 3 variables; Constraint 1; Step 4 lateness constraints; Step 5 objective; Step 7 output",
        "model_change": "Force assignment and add lateness variables",
        "data_needed": "reuse deadline_hr as soft target unless real data adds another field",
        "status": "Runnable demo in this notebook",
    },
    {
        "extension": "Hard vs soft deadline_hr",
        "where_to_change_in_01": "Step 3 feasible assignment filter; Step 5 objective; Step 7 output",
        "model_change": "Either block late choices or allow them with a penalty",
        "data_needed": "deadline_hr already exists",
        "status": "Policy comparison table included",
    },
    {
        "extension": "Recipe compatibility",
        "where_to_change_in_01": "Prepare scenario; Step 2 candidate runs; Step 3 variable filter; Constraint 3",
        "model_change": "Allow only compatible parts in the same run",
        "data_needed": "recipe_id or compatibility matrix",
        "status": "Exact-match compatibility table included",
    },
    {
        "extension": "Material availability",
        "where_to_change_in_01": "Prepare scenario; Step 3 variable filter; Step 7 output",
        "model_change": "Shift earliest feasible start time",
        "data_needed": "material_ready_hr",
        "status": "Demo data transform included",
    },
    {
        "extension": "Capacity by footprint",
        "where_to_change_in_01": "Prepare scenario; Constraint 2; Step 7 output",
        "model_change": "Replace part count capacity with area capacity",
        "data_needed": "part area_m2, autoclave usable_area_m2",
        "status": "Runnable capacity preview included",
    },
    {
        "extension": "Setup/changeover time",
        "where_to_change_in_01": "Step 2 candidate runs; Step 4 constraints; Step 7 output",
        "model_change": "Add sequencing relationship between recipe runs",
        "data_needed": "setup time matrix",
        "status": "Demo setup matrix included",
    },
    {
        "extension": "Maintenance downtime",
        "where_to_change_in_01": "Step 2 candidate runs or Step 4 constraints; Step 7 output",
        "model_change": "Block unavailable autoclave runs",
        "data_needed": "downtime windows",
        "status": "Runnable candidate-run filter included",
    },
    {
        "extension": "Staging space",
        "where_to_change_in_01": "Prepare scenario; Step 4 constraints; Step 7 output",
        "model_change": "Add time-bucket staging capacity",
        "data_needed": "staging capacity and staging timing assumptions",
        "status": "Usage preview included using soft-deadline model schedule",
    },
    {
        "extension": "Labor or crew availability",
        "where_to_change_in_01": "Prepare scenario; Step 4 constraints; Step 7 output",
        "model_change": "Add time-bucket labor capacity",
        "data_needed": "crew availability and crew required per run",
        "status": "Usage preview included using soft-deadline model schedule",
    },
    {
        "extension": "Objective swaps",
        "where_to_change_in_01": "Step 5 objective; Step 7 metrics",
        "model_change": "Change what best means",
        "data_needed": "depends on objective",
        "status": "Objective menu included",
    },
])

display(extensions)

## Recommended Order Later

The second-autoclave extension is already part of notebook `01`.

When we are ready to build beyond that, the clean order is probably:

1. Add explicit recipe compatibility data.
2. Add capacity by footprint.
3. Build the soft-deadline extension using `deadline_hr` as the soft lateness target.
4. Discuss setup time, downtime, staging, labor, and objective swaps.

This keeps the hands-on material understandable while still showing how the model can grow.